# Construction PPE Detection — SSDLite (MobileNetV3)

A deliberately **non-YOLO** detector, chosen to be the same size as the nano YOLOs so the
comparison is about *architecture*, not capacity.

**SSDLite320-MobileNetV3-Large** (torchvision): 3.44 M parameters as shipped, and
**2.35 M** once the classification head is resized from COCO's 91 classes to our 12
(11 PPE + background). That is *smaller* than YOLO26n's 2.57 M.

| | YOLO26n | YOLO11n | **SSDLite** |
|---|---|---|---|
| Params | 2.57 M | 2.62 M | **2.35 M** |
| Family | YOLO (anchor-free) | YOLO (anchor-free) | **SSD (anchor-based)** |
| Backbone | CSP-style CNN | C3k2 + C2PSA | **MobileNetV3-Large** |
| Input | 640 px | 640 px | **320 px** |
| Framework | Ultralytics | Ultralytics | **torchvision** |

### What is different about this notebook

Notebooks 01 and 02 call `model.train()` and Ultralytics handles everything. torchvision
gives you the model and nothing else, so this notebook carries its own:

- dataset class that converts YOLO-format labels into torchvision's `boxes`/`labels` dicts
- training loop, optimiser and LR schedule
- mAP evaluation via `torchmetrics`, matching the COCO definition Ultralytics reports

Everything else is kept identical on purpose — same sections, same `GPU_AVAILABLE`
branching, same `og_model/` → `trained_model/` flow, same live per-epoch table — so the
three notebooks can be read side by side.

### Runs anywhere, works offline

| | GPU present | No GPU |
|---|---|---|
| Device | CUDA or MPS | CPU |
| Batch | 16–32 | 8 |
| Epochs | 59 | 10 (capped) |

Weights come from `download.pytorch.org`, **not GitHub** — which may matter if GitHub is
the thing blocked on your network. They are cached in `og_model/` either way, so after
one online run this notebook needs no network at all.

## 1. Imports and offline configuration

In [ ]:
import os, sys, time, json, platform, shutil, warnings, random
from pathlib import Path
from collections import Counter

# ---------------------------------------------------------------------------
# Set False on a machine with no internet. Nothing will be fetched; a missing
# file raises a clear error naming the exact path it expected.
ALLOW_DOWNLOADS = True
# ---------------------------------------------------------------------------

os.environ.setdefault("PYTORCH_ENABLE_MPS_FALLBACK", "1")
os.environ.setdefault("OMP_NUM_THREADS", "4")
os.environ.setdefault("MKL_NUM_THREADS", "4")
os.environ.setdefault("KMP_DUPLICATE_LIB_OK", "TRUE")
warnings.filterwarnings("ignore", message=".*does not have a deterministic implementation.*")
warnings.filterwarnings("ignore", category=FutureWarning)
warnings.filterwarnings("ignore", message=".*Converting a tensor with requires_grad.*")

import torch
import torch.nn as nn
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import cv2
import torchvision
from torch.utils.data import Dataset, DataLoader
from torchvision.models.detection import ssdlite320_mobilenet_v3_large
from torchvision.models.detection.ssdlite import SSDLiteClassificationHead
from torchvision.models.detection import _utils as det_utils
from torchmetrics.detection import MeanAveragePrecision
from functools import partial

%matplotlib inline
plt.rcParams["figure.dpi"] = 110
plt.rcParams["font.size"] = 9

SEED = 0
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)

print("Python      ", sys.version.split()[0])
print("PyTorch     ", torch.__version__)
print("torchvision ", torchvision.__version__)
print("OpenCV      ", cv2.__version__)
print()
print("Downloads allowed:", ALLOW_DOWNLOADS)

## 2. Is there a GPU?

Same single boolean as notebooks 01 and 02 — set once, read everywhere.

In [ ]:
def host_info():
    info = {"os": f"{platform.system()} {platform.release()}", "arch": platform.machine(),
            "cpu": platform.processor() or platform.machine(),
            "cores": os.cpu_count(), "ram_gb": None}
    try:
        if platform.system() == "Darwin":
            import subprocess
            q = lambda k: subprocess.check_output(["sysctl", "-n", k], text=True).strip()
            info["cpu"] = q("machdep.cpu.brand_string")
            info["ram_gb"] = round(int(q("hw.memsize")) / 1024**3)
        elif platform.system() == "Linux":
            for ln in Path("/proc/meminfo").read_text().splitlines():
                if ln.startswith("MemTotal"):
                    info["ram_gb"] = round(int(ln.split()[1]) / 1024**2); break
        elif platform.system() == "Windows":
            import ctypes
            class MS(ctypes.Structure):
                _fields_ = [("dwLength", ctypes.c_ulong), ("dwMemoryLoad", ctypes.c_ulong),
                            ("ullTotalPhys", ctypes.c_ulonglong), ("ullAvailPhys", ctypes.c_ulonglong),
                            ("ullTotalPageFile", ctypes.c_ulonglong), ("ullAvailPageFile", ctypes.c_ulonglong),
                            ("ullTotalVirtual", ctypes.c_ulonglong), ("ullAvailVirtual", ctypes.c_ulonglong),
                            ("ullAvailExtendedVirtual", ctypes.c_ulonglong)]
            st = MS(); st.dwLength = ctypes.sizeof(MS)
            ctypes.windll.kernel32.GlobalMemoryStatusEx(ctypes.byref(st))
            info["ram_gb"] = round(st.ullTotalPhys / 1024**3)
    except Exception:
        pass
    return info

HOST = host_info()
print("=" * 66); print("  HOST"); print("=" * 66)
print(f"  OS     : {HOST['os']} ({HOST['arch']})")
print(f"  CPU    : {HOST['cpu']}")
print(f"  Cores  : {HOST['cores']}")
print(f"  RAM    : {HOST.get('ram_gb') or '?'} GB")
print("=" * 66); print()

HAS_CUDA = torch.cuda.is_available()
HAS_MPS  = bool(getattr(torch.backends, "mps", None)) and torch.backends.mps.is_available()
GPU_AVAILABLE = HAS_CUDA or HAS_MPS

if GPU_AVAILABLE:
    if HAS_CUDA:
        GPU_TYPE, DEVICE = "cuda", torch.device("cuda:0")
        GPU_NAME = torch.cuda.get_device_name(0)
        GPU_MEM = torch.cuda.get_device_properties(0).total_memory / 1024**3
        print(f"  GPU FOUND : {GPU_NAME} ({GPU_MEM:.1f} GB VRAM)")
    else:
        GPU_TYPE, DEVICE = "mps", torch.device("mps")
        GPU_NAME = "Apple Silicon GPU (Metal)"
        GPU_MEM = (HOST.get("ram_gb") or 16) * 0.4
        print(f"  GPU FOUND : {GPU_NAME} (unified memory, ~{GPU_MEM:.1f} GB usable)")
    t0 = time.perf_counter()
    _ = torch.randn(2048, 2048, device=DEVICE) @ torch.randn(2048, 2048, device=DEVICE)
    torch.cuda.synchronize() if HAS_CUDA else torch.mps.synchronize()
    print(f"              verified: 2048x2048 matmul in {(time.perf_counter()-t0)*1000:.0f} ms")
else:
    GPU_TYPE, DEVICE, GPU_NAME, GPU_MEM = None, torch.device("cpu"), "none", 0
    print("  NO GPU    : running on CPU. SSDLite at 320px is genuinely usable on CPU —")
    print("              far more so than a 640px YOLO — but still expect several")
    print("              minutes per epoch. Settings are reduced below.")

print()
print(f"  GPU_AVAILABLE = {GPU_AVAILABLE}   ->   device '{DEVICE}'")

## 3. Folder layout

Same convention as notebooks 01 and 02: `og_model/` holds the pristine pretrained
checkpoint, `trained_model/` holds what training produces.

In [ ]:
PROJECT = Path.cwd()
if not (PROJECT / "scripts").exists() and (PROJECT.parent / "scripts").exists():
    PROJECT = PROJECT.parent

OG_MODEL_DIR = PROJECT / "og_model"
TRAINED_DIR  = PROJECT / "trained_model"
DATA_DIR     = PROJECT / "dataset"
DATA_ROOT    = DATA_DIR / "construction-ppe"
RUNS_DIR     = PROJECT / "runs"
PRED_ROOT    = PROJECT / "predictions"
EXPORT_DIR   = PROJECT / "exports"
RUN_NAME     = "ssdlite"

# ===========================================================================
#  TRAIN = False  ->  skip training and use the weights already committed to
#                     trained_model/. Validation, test predictions, speed and
#                     the live camera all still run. THIS IS THE DEFAULT.
#  TRAIN = True   ->  retrain from the COCO checkpoint in og_model/.
# ===========================================================================
TRAIN = False
# ===========================================================================
SAVE_DIR     = RUNS_DIR / RUN_NAME
PRED_DIR     = PRED_ROOT / RUN_NAME

for d in (OG_MODEL_DIR, TRAINED_DIR, DATA_DIR, RUNS_DIR, PRED_ROOT, EXPORT_DIR,
          SAVE_DIR, PRED_DIR):
    d.mkdir(parents=True, exist_ok=True)

for label, path in [("og_model", OG_MODEL_DIR), ("trained_model", TRAINED_DIR),
                    ("dataset", DATA_DIR), ("runs", SAVE_DIR),
                    ("predictions", PRED_DIR), ("exports", EXPORT_DIR)]:
    print(f"  {label:<14} {path}")

## 4. The pretrained checkpoint

torchvision caches downloads in `~/.cache/torch`, which does not travel with the project.
We copy the file into `og_model/` and load from there, so the notebook works on a machine
with no network.

In [ ]:
MODEL_NAME = "ssdlite320_mobilenet_v3_large.pth"
OG_WEIGHTS = OG_MODEL_DIR / MODEL_NAME

if OG_WEIGHTS.exists():
    print(f"Found local checkpoint: {OG_WEIGHTS}")
    print(f"  size: {OG_WEIGHTS.stat().st_size/1024**2:.2f} MB — no download needed")
    base_state = torch.load(OG_WEIGHTS, map_location="cpu", weights_only=True)
    model = ssdlite320_mobilenet_v3_large(weights=None, weights_backbone=None)
    model.load_state_dict(base_state)

elif ALLOW_DOWNLOADS:
    print("Downloading COCO-pretrained SSDLite (13 MB, from download.pytorch.org)...")
    model = ssdlite320_mobilenet_v3_large(weights="DEFAULT")
    torch.save(model.state_dict(), OG_WEIGHTS)
    print(f"  cached -> {OG_WEIGHTS} ({OG_WEIGHTS.stat().st_size/1024**2:.2f} MB)")
    print("  future runs load from here and need no network.")

else:
    raise FileNotFoundError(
        f"\n{'='*70}\n"
        f"  {MODEL_NAME} is missing and ALLOW_DOWNLOADS is False.\n\n"
        f"  Expected at: {OG_WEIGHTS}\n\n"
        f"  Run this notebook once on a networked machine to populate og_model/,\n"
        f"  then copy that folder across.\n{'='*70}")

n_pre = sum(p.numel() for p in model.parameters())
print(f"\nPretrained (91 COCO classes): {n_pre:,} params ({n_pre/1e6:.2f} M)")

## 5. The dataset

The images are shared with notebooks 01 and 02 — nothing is duplicated.

In [ ]:
CLASS_NAMES = {0: "helmet", 1: "gloves", 2: "vest", 3: "boots", 4: "goggles",
               5: "none", 6: "Person", 7: "no_helmet", 8: "no_goggle",
               9: "no_gloves", 10: "no_boots"}
IMG_EXT = {".jpg", ".jpeg", ".png", ".bmp", ".webp"}

# torchvision reserves label 0 for background, so every PPE class shifts up by one.
NUM_CLASSES = len(CLASS_NAMES) + 1          # 11 + background
IDX_TO_NAME = {i + 1: n for i, n in CLASS_NAMES.items()}

if (DATA_ROOT / "images" / "train").exists():
    print(f"Dataset found: {DATA_ROOT}")
elif ALLOW_DOWNLOADS:
    print("Dataset missing — downloading (~170 MB)...")
    import urllib.request, zipfile, tempfile
    URL = "https://github.com/ultralytics/assets/releases/download/v0.0.0/construction-ppe.zip"
    with tempfile.TemporaryDirectory() as tmp:
        z = Path(tmp) / "ppe.zip"
        urllib.request.urlretrieve(URL, z)
        with zipfile.ZipFile(z) as zf:
            zf.extractall(DATA_ROOT)
    inner = DATA_ROOT / "construction-ppe"
    if inner.exists():
        for it in inner.iterdir():
            it.rename(DATA_ROOT / it.name)
        inner.rmdir()
    print("  done.")
else:
    raise FileNotFoundError(f"Dataset missing at {DATA_ROOT} and ALLOW_DOWNLOADS is False.")

split_counts = {}
for split in ("train", "val", "test"):
    imgs = sorted(p for p in (DATA_ROOT/"images"/split).glob("*") if p.suffix.lower() in IMG_EXT)
    split_counts[split] = len(imgs)
    print(f"  {split:<6} {len(imgs):>5} images")
print(f"  {'total':<6} {sum(split_counts.values()):>5} images   (expected 1416)")
assert sum(split_counts.values()) > 0
print(f"\nLabel mapping: background=0, {IDX_TO_NAME[1]}=1 ... {IDX_TO_NAME[11]}=11")

## 6. Look at the data before training on it

Identical analysis to notebooks 01 and 02 — the class-frequency chart is what predicts
which classes will end up weak.

In [ ]:
counts, boxes_per_image = Counter(), []
for lbl in (DATA_ROOT/"labels"/"train").glob("*.txt"):
    lines = [l for l in lbl.read_text().strip().splitlines() if l.strip()]
    boxes_per_image.append(len(lines))
    for line in lines:
        counts[int(line.split()[0])] += 1

dist = (pd.DataFrame([(CLASS_NAMES[c], n) for c, n in counts.items()],
                     columns=["class","instances"])
        .sort_values("instances", ascending=False).reset_index(drop=True))
print(f"Labelled boxes : {sum(counts.values()):,}")
print(f"Imbalance      : {dist['instances'].max()/dist['instances'].min():.0f}:1\n")
display(dist)

fig, axes = plt.subplots(1, 2, figsize=(14, 4.5))
colors = ["#d62728" if c.startswith("no_") else "#2ca02c" for c in dist["class"]]
axes[0].barh(dist["class"], dist["instances"], color=colors); axes[0].invert_yaxis()
axes[0].set_xlabel("training instances"); axes[0].set_title("Class frequency (red = 'missing PPE')")
for i, v in enumerate(dist["instances"]):
    axes[0].text(v + max(dist["instances"])*0.01, i, str(v), va="center", fontsize=8)
axes[1].hist(boxes_per_image, bins=range(0, max(boxes_per_image)+2), color="#4C78A8")
axes[1].set_xlabel("boxes per image"); axes[1].set_title("How crowded are the images?")
plt.tight_layout(); plt.show()

> Same picture as the other two notebooks: the `no_*` classes are rare, and they are the
> ones a safety system exists to catch. YOLO26n and YOLO11n both scored 0.03–0.13 on
> them. If SSDLite — a different architecture family entirely — lands in the same place,
> that is three independent models agreeing the limit is the **data**, not the model.

## 7. Dataset class

This is the part Ultralytics does invisibly. YOLO labels are
`class cx cy w h`, all normalised 0–1; torchvision wants absolute `[x1,y1,x2,y2]` pixel
boxes plus 1-indexed labels.

Three details that cause silent breakage if missed:

- **Degenerate boxes.** Any box where `x2 <= x1` or `y2 <= y1` makes the SSD loss produce
  NaN. They are filtered out rather than clamped, so a bad label is dropped, not distorted.
- **Empty images.** An image whose boxes all get filtered still has to return correctly
  *shaped* empty tensors — `(0,4)` float and `(0,)` int64 — or collation fails.
- **Horizontal flip.** The only augmentation used, and the box x-coordinates must flip
  with the image. Getting this backwards trains the model on systematically wrong boxes.

In [ ]:
class PPEDataset(Dataset):
    def __init__(self, root: Path, split: str, imgsz: int = 320, train: bool = False):
        self.img_dir = root / "images" / split
        self.lbl_dir = root / "labels" / split
        self.imgsz = imgsz
        self.train = train
        self.items = sorted(p for p in self.img_dir.glob("*") if p.suffix.lower() in IMG_EXT)

    def __len__(self):
        return len(self.items)

    def __getitem__(self, i):
        path = self.items[i]
        img = cv2.imread(str(path))
        img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
        h0, w0 = img.shape[:2]
        img = cv2.resize(img, (self.imgsz, self.imgsz), interpolation=cv2.INTER_LINEAR)

        boxes, labels = [], []
        lbl = self.lbl_dir / f"{path.stem}.txt"
        if lbl.exists():
            for line in lbl.read_text().strip().splitlines():
                parts = line.split()
                if len(parts) < 5:
                    continue
                cls = int(parts[0])
                cx, cy, bw, bh = (float(v) for v in parts[1:5])
                # normalised cxcywh -> absolute xyxy in the RESIZED frame
                x1 = (cx - bw/2) * self.imgsz
                y1 = (cy - bh/2) * self.imgsz
                x2 = (cx + bw/2) * self.imgsz
                y2 = (cy + bh/2) * self.imgsz
                x1, y1 = max(0.0, x1), max(0.0, y1)
                x2, y2 = min(float(self.imgsz), x2), min(float(self.imgsz), y2)
                if x2 - x1 < 1.0 or y2 - y1 < 1.0:
                    continue                      # degenerate -> would NaN the loss
                boxes.append([x1, y1, x2, y2])
                labels.append(cls + 1)            # shift for background=0

        if self.train and random.random() < 0.5:  # horizontal flip
            img = np.ascontiguousarray(img[:, ::-1])
            boxes = [[self.imgsz - x2, y1, self.imgsz - x1, y2] for x1, y1, x2, y2 in boxes]

        t = torch.from_numpy(img).permute(2, 0, 1).float() / 255.0
        target = {
            "boxes":  torch.tensor(boxes, dtype=torch.float32) if boxes
                      else torch.zeros((0, 4), dtype=torch.float32),
            "labels": torch.tensor(labels, dtype=torch.int64) if labels
                      else torch.zeros((0,), dtype=torch.int64),
            "image_id": torch.tensor([i]),
            "orig_size": torch.tensor([h0, w0]),
            "path": str(path),
        }
        return t, target


def collate(batch):
    return tuple(zip(*batch))

# quick sanity check before building the loaders
_ds = PPEDataset(DATA_ROOT, "train", train=True)
_img, _tgt = _ds[0]
print(f"dataset size : {len(_ds)}")
print(f"image tensor : {tuple(_img.shape)}  range [{_img.min():.2f}, {_img.max():.2f}]")
print(f"boxes        : {tuple(_tgt['boxes'].shape)}  labels {_tgt['labels'].tolist()[:8]}")
assert _tgt["boxes"].shape[1] == 4
assert (_tgt["boxes"][:, 2] > _tgt["boxes"][:, 0]).all(), "degenerate box survived"
assert (_tgt["boxes"][:, 3] > _tgt["boxes"][:, 1]).all(), "degenerate box survived"
print("sanity checks passed")

In [ ]:
# Show the dataset's own output, boxes included — proof the conversion is right.
fig, axes = plt.subplots(2, 3, figsize=(15, 8.5))
_vis = PPEDataset(DATA_ROOT, "train", train=False)
for ax, idx in zip(axes.ravel(), [0, 120, 400, 700, 900, 1050]):
    if idx >= len(_vis):
        ax.axis("off"); continue
    im, tg = _vis[idx]
    arr = (im.permute(1, 2, 0).numpy() * 255).astype(np.uint8).copy()
    palette = plt.cm.tab20(np.linspace(0, 1, 12))[:, :3] * 255
    for (x1, y1, x2, y2), lb in zip(tg["boxes"].tolist(), tg["labels"].tolist()):
        col = tuple(int(c) for c in palette[lb])
        cv2.rectangle(arr, (int(x1), int(y1)), (int(x2), int(y2)), col, 2)
        cv2.putText(arr, IDX_TO_NAME[lb], (int(x1), max(int(y1)-5, 11)),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.42, col, 1)
    ax.imshow(arr); ax.set_title(f"{Path(tg['path']).name} — {len(tg['labels'])} boxes", fontsize=9)
    ax.axis("off")
plt.suptitle("Dataset output at 320px — converted from YOLO labels", fontsize=12)
plt.tight_layout(); plt.show()

## 8. Resize the head, and set the training config

The COCO model predicts 91 classes. We swap the classification head for a 12-class one
(11 PPE + background) and keep the pretrained MobileNetV3 backbone — the features it
learned on COCO transfer; only the final classifier has to be relearned.

That swap is also why the parameter count *drops*: 91 classes worth of output channels
become 12.

In [ ]:
IMGSZ = 320                       # SSDLite320 is designed for this input size

if GPU_AVAILABLE:
    # ---------------- GPU PATH ----------------
    EPOCHS  = 59
    if GPU_TYPE == "cuda":
        BATCH   = 32 if GPU_MEM >= 16 else 16
        WORKERS = min(8, os.cpu_count() or 4)
    else:                          # mps
        BATCH   = 16
        WORKERS = 0                # loader processes are slower than inline on MPS
    LR          = 1e-3
    BENCH_RUNS  = 30
    print("GPU PATH")
    print(f"  device  : {DEVICE} ({GPU_TYPE})")
else:
    # ---------------- CPU PATH ----------------
    EPOCHS      = 10               # capped; 59 on CPU would take hours
    BATCH       = 8
    WORKERS     = 0
    LR          = 1e-3
    BENCH_RUNS  = 8
    print("!" * 68)
    print("  CPU PATH — no GPU detected")
    print("!" * 68)
    print("  SSDLite at 320px is the most CPU-friendly of the three models here,")
    print("  but still expect a few minutes per epoch.")
    print(f"  Epochs capped at {EPOCHS} (from 59). Set EPOCHS = 1 first if you")
    print("  just want to confirm the pipeline runs.")

print(f"  imgsz   : {IMGSZ}")
print(f"  batch   : {BATCH}")
print(f"  workers : {WORKERS}")
print(f"  epochs  : {EPOCHS}")
print(f"  lr      : {LR}")
print()

# --- swap the classification head for our class count --------------------
in_channels = det_utils.retrieve_out_channels(model.backbone, (IMGSZ, IMGSZ))
num_anchors = model.anchor_generator.num_anchors_per_location()
norm_layer  = partial(nn.BatchNorm2d, eps=0.001, momentum=0.03)
model.head.classification_head = SSDLiteClassificationHead(
    in_channels, num_anchors, NUM_CLASSES, norm_layer)

n_params = sum(p.numel() for p in model.parameters())
print(f"head swapped: 91 COCO classes -> {NUM_CLASSES} ({NUM_CLASSES-1} PPE + background)")
print(f"  in_channels : {in_channels}")
print(f"  anchors/loc : {num_anchors}")
print(f"  parameters  : {n_params:,} ({n_params/1e6:.2f} M)   was {n_pre/1e6:.2f} M")

model.to(DEVICE)

train_ds = PPEDataset(DATA_ROOT, "train", IMGSZ, train=True)
val_ds   = PPEDataset(DATA_ROOT, "val",   IMGSZ, train=False)
test_ds  = PPEDataset(DATA_ROOT, "test",  IMGSZ, train=False)

train_loader = DataLoader(train_ds, batch_size=BATCH, shuffle=True,
                          num_workers=WORKERS, collate_fn=collate)
val_loader   = DataLoader(val_ds, batch_size=BATCH, shuffle=False,
                          num_workers=WORKERS, collate_fn=collate)

print(f"\nbatches: {len(train_loader)} train / {len(val_loader)} val")

## 9. Train

AdamW at 1e-3 with a cosine schedule. The same reasoning as notebooks 01/02 applies —
AdamW wants a learning rate around 1e-3, and the `lr0=0.01` figure that circulates for
these models is an order of magnitude too hot for it.

One torchvision-specific thing worth knowing: the model returns **a dict of losses in
`train()` mode** and **a list of predictions in `eval()` mode**. Same object, different
output depending on mode — forgetting to switch is the classic cause of "why is my loss
a list of boxes?"

The table below fills in one row per epoch, exactly like the other two notebooks.

In [ ]:
params = [p for p in model.parameters() if p.requires_grad]
optimizer = torch.optim.AdamW(params, lr=LR, weight_decay=5e-4)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=EPOCHS, eta_min=LR*0.01)

metric = MeanAveragePrecision(box_format="xyxy", iou_type="bbox", class_metrics=True)

def _eta(sec):
    sec = int(max(sec, 0)); h, rem = divmod(sec, 3600); m, s = divmod(rem, 60)
    return f"{h}h{m:02d}m" if h else f"{m}m{s:02d}s"

LOGS_DIR = PROJECT / "training_logs"
history, epoch_times = [], []
best = {"map": -1.0, "epoch": 0}
train_t0 = time.monotonic()

if not TRAIN:
    # ---------------- inference mode (the default) -------------------------
    BEST = TRAINED_DIR / f"{RUN_NAME}_best.pt"
    if not BEST.exists():
        raise FileNotFoundError(chr(10).join([
            "=" * 70,
            "  Trained weights not found:",
            f"    {BEST}",
            "",
            "  Re-clone the repository (the weights are committed there), or set",
            "  TRAIN = True in the cell above to train them yourself.",
            "=" * 70]))
    model.load_state_dict(torch.load(BEST, map_location=DEVICE, weights_only=True))
    model.to(DEVICE).eval()
    elapsed = 0.0
    log = LOGS_DIR / f"{RUN_NAME}_results.csv"
    if log.exists():
        history = pd.read_csv(log).to_dict(orient="records")
        best = {"map": max(h["mAP50-95"] for h in history),
                "epoch": int(max(history, key=lambda h: h["mAP50-95"])["epoch"])}
    print("TRAIN = False  ->  using the committed weights. No training run.")
    print(f"  weights : {BEST}")
    print(f"  size    : {BEST.stat().st_size/1024**2:.2f} MB")
    if history:
        print(f"  epoch log loaded: {len(history)} epochs, "
              f"best mAP50-95 {best['map']:.4f} at epoch {best['epoch']}")
    print()
    print("  Set TRAIN = True above to retrain from scratch.")

else:
    print(f"Training {RUN_NAME}: {EPOCHS} epochs on {DEVICE} (imgsz {IMGSZ}, batch {BATCH})")
    print("=" * 104)
    print(f"{'epoch':>7} {'box':>8} {'cls':>8} {'total':>8} | {'mAP50':>7} {'mAP50-95':>9} "
          f"| {'lr':>8} {'sec':>6} {'elapsed':>8} {'eta':>8}")
    print("=" * 104, flush=True)

    for epoch in range(1, EPOCHS + 1):
        t_ep = time.monotonic()

        # ---- train ---------------------------------------------------------
        model.train()
        sums = {"bbox_regression": 0.0, "classification": 0.0}
        nb = 0
        for images, targets in train_loader:
            images = [im.to(DEVICE) for im in images]
            tg = [{"boxes": t["boxes"].to(DEVICE), "labels": t["labels"].to(DEVICE)}
                  for t in targets]
            losses = model(images, tg)
            total = sum(losses.values())
            if not torch.isfinite(total):
                print(f"   [warn] non-finite loss at epoch {epoch}, batch {nb} — skipped", flush=True)
                optimizer.zero_grad(set_to_none=True); continue
            optimizer.zero_grad(set_to_none=True)
            total.backward()
            torch.nn.utils.clip_grad_norm_(params, 10.0)     # SSD losses can spike early
            optimizer.step()
            for k in sums:
                sums[k] += float(losses[k].detach())
            nb += 1
        scheduler.step()
        box_l = sums["bbox_regression"] / max(nb, 1)
        cls_l = sums["classification"] / max(nb, 1)

        # ---- validate ------------------------------------------------------
        model.eval()
        metric.reset()
        with torch.no_grad():
            for images, targets in val_loader:
                imgs = [im.to(DEVICE) for im in images]
                outs = model(imgs)
                # torchmetrics wants CPU tensors
                preds = [{"boxes": o["boxes"].detach().cpu(),
                          "scores": o["scores"].detach().cpu(),
                          "labels": o["labels"].detach().cpu()} for o in outs]
                gts   = [{"boxes": t["boxes"], "labels": t["labels"]} for t in targets]
                metric.update(preds, gts)
        res = metric.compute()
        m50, m95 = float(res["map_50"]), float(res["map"])

        dt = time.monotonic() - t_ep
        epoch_times.append(dt)
        elapsed = time.monotonic() - train_t0
        lr_now = optimizer.param_groups[0]["lr"]

        star = ""
        if m95 > best["map"]:
            best.update({"map": m95, "epoch": epoch}); star = "  <-- best"
            torch.save(model.state_dict(), SAVE_DIR / "best.pt")
        torch.save(model.state_dict(), SAVE_DIR / "last.pt")

        history.append({"epoch": epoch, "box_loss": box_l, "cls_loss": cls_l,
                        "total_loss": box_l + cls_l, "mAP50": m50, "mAP50-95": m95,
                        "lr": lr_now, "seconds": dt,
                        "map_per_class": [float(x) for x in res.get("map_per_class", [])]
                                          if res.get("map_per_class") is not None else []})
        pd.DataFrame(history).to_csv(SAVE_DIR / "results.csv", index=False)

        print(f"{epoch:>3}/{EPOCHS:<3} {box_l:>8.4f} {cls_l:>8.4f} {box_l+cls_l:>8.4f} "
              f"| {m50:>7.4f} {m95:>9.4f} | {lr_now:>8.2e} {dt:>6.1f} "
              f"{_eta(elapsed):>8} {_eta((elapsed/epoch)*(EPOCHS-epoch)):>8}{star}", flush=True)

    elapsed = time.monotonic() - train_t0
    BEST = SAVE_DIR / "best.pt"
    print("=" * 104)
    print(f"Best mAP50-95 {best['map']:.4f} at epoch {best['epoch']}")
    print(f"\nFinished in {elapsed/60:.1f} min ({np.mean(epoch_times):.0f}s per epoch on {DEVICE})")
    print(f"Weights: {BEST} ({BEST.stat().st_size/1024**2:.2f} MB)")

## 10. Save the trained weights to `trained_model/`

In [ ]:
saved = {}
if not TRAIN:
    for kind in ("best", "last"):
        dst = TRAINED_DIR / f"{RUN_NAME}_{kind}.pt"
        if dst.exists():
            saved[kind] = str(dst)
            print(f"  {kind:<5} already in trained_model/  "
                  f"({dst.stat().st_size/1024**2:.2f} MB)")
    print()
    print("(TRAIN = False, so nothing was retrained and nothing overwritten.)")

for kind in ("best", "last") if TRAIN else ():
    src = SAVE_DIR / f"{kind}.pt"
    if src.exists():
        dst = TRAINED_DIR / f"{RUN_NAME}_{kind}.pt"
        shutil.copy2(src, dst)
        saved[kind] = str(dst)
        print(f"  {kind:<5} -> {dst}   ({dst.stat().st_size/1024**2:.2f} MB)")

print(f"\nog_model/      : {MODEL_NAME} (pretrained, untouched)")
print(f"trained_model/ : {', '.join(Path(v).name for v in saved.values())}")

## 11. Every epoch, in full

In [ ]:
hist = pd.DataFrame(history)
if hist.empty:
    raise RuntimeError("No epoch history. Check training_logs/ssdlite_results.csv exists.")
table = hist[["epoch","box_loss","cls_loss","total_loss","mAP50","mAP50-95","lr","seconds"]].copy()
for c in ["box_loss","cls_loss","total_loss","mAP50","mAP50-95"]:
    table[c] = table[c].round(4)
table["lr"] = table["lr"].map(lambda x: f"{x:.2e}")
table["seconds"] = table["seconds"].round(1)
table["best"] = ["  <-- best" if e == best["epoch"] else "" for e in table["epoch"]]

print(f"All {len(table)} epochs (best mAP50-95 at epoch {best['epoch']})")
print("=" * 100)
print(table.to_string(index=False))
print("=" * 100)

---

## 12. Validation

**This section and everything after it runs on its own.** It needs only the trained
weights in `trained_model/` and the dataset — no training, and no dependency on the
cells above having been run.

The cell below rebuilds the handful of names the later cells use. Run it after a kernel
restart, or if you jumped straight here, and the rest of the notebook will work. It is
harmless to run when the earlier cells already executed: anything already defined is
left untouched.

In [ ]:
# ---- makes this section self-contained ------------------------------------
# setdefault, not assignment: if the cells above already ran, their values win.
import ppe_lib as ppe
for _k, _v in ppe.bootstrap(RUN_NAME if "RUN_NAME" in dir() else "ssdlite").items():
    globals().setdefault(_k, _v)

In [ ]:
model.load_state_dict(torch.load(BEST, map_location=DEVICE, weights_only=True))
model.to(DEVICE).eval()

metric.reset()
with torch.no_grad():
    for images, targets in val_loader:
        outs = model([im.to(DEVICE) for im in images])
        metric.update([{"boxes": o["boxes"].cpu(), "scores": o["scores"].cpu(),
                        "labels": o["labels"].cpu()} for o in outs],
                      [{"boxes": t["boxes"], "labels": t["labels"]} for t in targets])
res = metric.compute()

summary = {"mAP50": float(res["map_50"]), "mAP50-95": float(res["map"]),
           "mAP_small": float(res["map_small"]), "mAP_medium": float(res["map_medium"]),
           "mAP_large": float(res["map_large"])}
print("Validation (143 held-out images)")
print("-" * 46)
for k, v in summary.items():
    print(f"  {k:<11} {v:.4f}" if v >= 0 else f"  {k:<11} n/a")

In [ ]:
# Per-class. torchmetrics returns classes in the order of `classes`, not 1..11,
# so they must be zipped rather than assumed contiguous.
per_cls  = res.get("map_per_class")
cls_ids  = res.get("classes")
rows = []
if per_cls is not None and cls_ids is not None:
    for cid, v in zip(cls_ids.tolist(), per_cls.tolist()):
        name = IDX_TO_NAME.get(int(cid), str(cid))
        rows.append({"class": name, "mAP50-95": max(float(v), 0.0),
                     "train_instances": counts.get(int(cid) - 1, 0)})
per_class = pd.DataFrame(rows).sort_values("mAP50-95", ascending=False).reset_index(drop=True)
display(per_class.style.format({"mAP50-95": "{:.4f}"})
        .background_gradient(cmap="RdYlGn", subset=["mAP50-95"], vmin=0, vmax=0.6))

fig, axes = plt.subplots(1, 2, figsize=(15, 4.6))
cols = ["#d62728" if c.startswith("no_") else "#2ca02c" for c in per_class["class"]]
axes[0].barh(per_class["class"], per_class["mAP50-95"], color=cols); axes[0].invert_yaxis()
axes[0].set_xlabel("mAP50-95"); axes[0].set_title("Per-class accuracy (red = 'missing PPE')")
axes[0].grid(axis="x", alpha=.3)
axes[1].scatter(per_class["train_instances"], per_class["mAP50-95"], c=cols, s=90,
                edgecolors="black", linewidths=.5, zorder=3)
for _, r in per_class.iterrows():
    axes[1].annotate(r["class"], (r["train_instances"], r["mAP50-95"]),
                     fontsize=7.5, xytext=(4,4), textcoords="offset points")
axes[1].set_xscale("symlog"); axes[1].set_xlabel("training instances (log)")
axes[1].set_ylabel("mAP50-95"); axes[1].set_title("Accuracy vs how often the class was seen")
axes[1].grid(alpha=.3)
plt.tight_layout(); plt.show()

## 13. Predictions — original beside model output

In [ ]:
CONF = 0.30
PRED_DIR.mkdir(parents=True, exist_ok=True)

def predict_one(idx, ds=test_ds):
    img_t, tgt = ds[idx]
    with torch.no_grad():
        out = model([img_t.to(DEVICE)])[0]
    keep = out["scores"] >= CONF
    return (img_t, tgt,
            out["boxes"][keep].cpu().numpy(),
            out["scores"][keep].cpu().numpy(),
            out["labels"][keep].cpu().numpy())

def draw(img_t, boxes, labels, scores):
    arr = (img_t.permute(1,2,0).numpy()*255).astype(np.uint8).copy()
    palette = plt.cm.tab20(np.linspace(0,1,12))[:, :3]*255
    for (x1,y1,x2,y2), lb, sc in zip(boxes, labels, scores):
        col = tuple(int(c) for c in palette[int(lb)])
        cv2.rectangle(arr, (int(x1),int(y1)), (int(x2),int(y2)), col, 2)
        cv2.putText(arr, f"{IDX_TO_NAME[int(lb)]} {sc:.2f}", (int(x1), max(int(y1)-5,11)),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.42, col, 1)
    return arr

# run over the whole test split and save annotated frames + label files
all_preds, total_boxes, empty = [], 0, 0
(PRED_DIR / "labels").mkdir(exist_ok=True)
for i in range(len(test_ds)):
    img_t, tgt, b, s, l = predict_one(i)
    all_preds.append((img_t, tgt, b, s, l))
    total_boxes += len(b); empty += (len(b) == 0)
    name = Path(tgt["path"]).stem
    cv2.imwrite(str(PRED_DIR / f"{name}.jpg"),
                cv2.cvtColor(draw(img_t, b, l, s), cv2.COLOR_RGB2BGR))
    with open(PRED_DIR / "labels" / f"{name}.txt", "w") as f:
        for (x1,y1,x2,y2), lb, sc in zip(b, l, s):
            cx, cy = (x1+x2)/2/IMGSZ, (y1+y2)/2/IMGSZ
            bw, bh = (x2-x1)/IMGSZ, (y2-y1)/IMGSZ
            f.write(f"{int(lb)-1} {cx:.6f} {cy:.6f} {bw:.6f} {bh:.6f} {sc:.4f}\n")

print(f"Images          : {len(all_preds)}")
print(f"Detections      : {total_boxes}  ({total_boxes/max(len(all_preds),1):.1f} per image)")
print(f"Images with none: {empty}   (conf threshold {CONF})")
print(f"Saved to        : {PRED_DIR}")

In [ ]:
show = all_preds[:5]
fig, axes = plt.subplots(len(show), 2, figsize=(13, 4.6*len(show)))
if len(show) == 1:
    axes = np.array([axes])
for row, (img_t, tgt, b, s, l) in zip(axes, show):
    orig = (img_t.permute(1,2,0).numpy()*255).astype(np.uint8)
    row[0].imshow(orig); row[0].set_title(f"Original — {Path(tgt['path']).name}", fontsize=10)
    row[0].axis("off")
    row[1].imshow(draw(img_t, b, l, s))
    if len(b):
        found = Counter(IDX_TO_NAME[int(x)] for x in l)
        t = ", ".join(f"{k} x{v}" for k,v in found.items()) + chr(10) + \
            f"conf {s.min():.2f}-{s.max():.2f}"
    else:
        t = "nothing detected"
    row[1].set_title(t, fontsize=10); row[1].axis("off")
plt.suptitle(f"SSDLite — test predictions at conf {CONF}", fontsize=13, y=0.999)
plt.tight_layout(); plt.show()

In [ ]:
pred_counts, confs = Counter(), []
for _, _, b, s, l in all_preds:
    for lb, sc in zip(l, s):
        pred_counts[IDX_TO_NAME[int(lb)]] += 1; confs.append(float(sc))

pdist = (pd.DataFrame(pred_counts.items(), columns=["class","detections"])
         .sort_values("detections", ascending=False).reset_index(drop=True))
fig, axes = plt.subplots(1, 2, figsize=(14, 4.2))
cols = ["#d62728" if c.startswith("no_") else "#2ca02c" for c in pdist["class"]]
axes[0].barh(pdist["class"], pdist["detections"], color=cols); axes[0].invert_yaxis()
axes[0].set_xlabel("detections"); axes[0].set_title("What the model predicts")
if confs:
    axes[1].hist(confs, bins=30, color="#4C78A8", edgecolor="white")
    axes[1].axvline(CONF, ls="--", c="#d62728", label=f"threshold {CONF}")
    axes[1].legend(fontsize=8)
axes[1].set_xlabel("confidence"); axes[1].set_title("Confidence distribution")
plt.tight_layout(); plt.show()
display(pdist)
if confs:
    print(f"Median confidence: {np.median(confs):.3f}")

## 14. Speed

In [ ]:
sample = [test_ds[i][0] for i in range(min(10, len(test_ds)))]
model.eval()

def measure(imgsz_imgs, runs):
    for i in range(3):                                   # warmup
        with torch.no_grad():
            model([imgsz_imgs[i % len(imgsz_imgs)].to(DEVICE)])
    if GPU_AVAILABLE:
        torch.cuda.synchronize() if GPU_TYPE == "cuda" else torch.mps.synchronize()
    ts = []
    for i in range(runs):
        t0 = time.perf_counter()
        with torch.no_grad():
            model([imgsz_imgs[i % len(imgsz_imgs)].to(DEVICE)])
        if GPU_AVAILABLE:
            torch.cuda.synchronize() if GPU_TYPE == "cuda" else torch.mps.synchronize()
        ts.append((time.perf_counter()-t0)*1000)
    return ts

t = measure(sample, BENCH_RUNS)
bench = {IMGSZ: {"mean_ms": float(np.mean(t)), "median_ms": float(np.median(t)),
                 "p90_ms": float(np.percentile(t,90)), "fps": 1000/float(np.mean(t))}}
print(f"  {IMGSZ}px  {np.mean(t):7.2f} ms   {1000/np.mean(t):7.1f} FPS on {DEVICE}")

fig, ax = plt.subplots(figsize=(6,4))
f = bench[IMGSZ]["fps"]
ax.bar([f"{IMGSZ}px"], [f], color="#2ca02c" if f>=30 else "#d62728", width=.5)
ax.axhline(30, ls="--", c="black", lw=1.2, label="30 FPS realtime")
ax.text(0, f+max(f,30)*0.02, f"{f:.0f}", ha="center", fontsize=10)
ax.set_ylabel("FPS"); ax.set_title(f"SSDLite throughput on {DEVICE}")
ax.legend(fontsize=8); ax.grid(axis="y", alpha=.3)
plt.tight_layout(); plt.show()
print(f"{'YES' if f>=30 else 'NO'} — {f/30:.1f}x the 30 FPS bar")

## 15. Export

In [ ]:
exports = {}
model.eval()
dummy = torch.randn(1, 3, IMGSZ, IMGSZ, device=DEVICE)

try:
    p = EXPORT_DIR / f"{RUN_NAME}.onnx"
    torch.onnx.export(model, [dummy[0]], str(p), opset_version=13,
                      input_names=["images"], do_constant_folding=True)
    exports["onnx"] = {"path": str(p), "size_mb": round(p.stat().st_size/1024**2, 2)}
except Exception as e:
    exports["onnx"] = {"error": str(e)[:160]}
    print("ONNX export failed:", str(e)[:160])

try:
    p = EXPORT_DIR / f"{RUN_NAME}_state_dict.pt"
    torch.save(model.state_dict(), p)
    exports["state_dict"] = {"path": str(p), "size_mb": round(p.stat().st_size/1024**2, 2)}
except Exception as e:
    exports["state_dict"] = {"error": str(e)[:160]}

print("\nArtifacts")
print("-" * 62)
print(f"  {'trained best.pt':<18} {BEST.stat().st_size/1024**2:>7.2f} MB")
for k, v in exports.items():
    if "size_mb" in v:
        print(f"  {k:<18} {v['size_mb']:>7.2f} MB   {v['path']}")
    else:
        print(f"  {k:<18} {'—':>7}      {v['error'][:60]}")

---

# 16. Final recap

In [ ]:
fig = plt.figure(figsize=(16, 8.5))
gs = fig.add_gridspec(2, 3, hspace=.32, wspace=.24)

ax = fig.add_subplot(gs[0,0])
ax.plot(hist["epoch"], hist["box_loss"], lw=1.8, label="box")
ax.plot(hist["epoch"], hist["cls_loss"], lw=1.8, label="classification")
ax.set_title("Training loss"); ax.set_xlabel("epoch"); ax.legend(fontsize=8); ax.grid(alpha=.3)

ax = fig.add_subplot(gs[0,1])
ax.plot(hist["epoch"], hist["total_loss"], lw=2, color="#d62728")
ax.set_title("Total loss"); ax.set_xlabel("epoch"); ax.grid(alpha=.3)

ax = fig.add_subplot(gs[0,2])
ax.plot(hist["epoch"], hist["lr"], lw=1.8, color="#8c564b")
ax.set_title("Learning rate (cosine)"); ax.set_xlabel("epoch"); ax.grid(alpha=.3)

ax = fig.add_subplot(gs[1,:2])
ax.plot(hist["epoch"], hist["mAP50"], lw=2.4, color="#2ca02c", label="mAP50")
ax.plot(hist["epoch"], hist["mAP50-95"], lw=2.4, color="#1f77b4", label="mAP50-95")
pk = hist["mAP50-95"].idxmax()
ax.scatter(hist.loc[pk,"epoch"], hist.loc[pk,"mAP50-95"], s=130, facecolors="none",
           edgecolors="#d62728", linewidths=2.2, zorder=5,
           label=f"best @ epoch {int(hist.loc[pk,'epoch'])}")
ax.set_title("Validation mAP — the number that matters", fontsize=11)
ax.set_xlabel("epoch"); ax.legend(fontsize=9); ax.grid(alpha=.3)

ax = fig.add_subplot(gs[1,2])
ax.plot(hist["epoch"], hist["seconds"], lw=1.6, marker="o", ms=3, color="#4C78A8")
ax.set_title("Seconds per epoch"); ax.set_xlabel("epoch"); ax.grid(alpha=.3)

fig.suptitle(f"SSDLite — {len(hist)} epochs on {DEVICE}", fontsize=14)
plt.show()
print(f"Best mAP50-95 : {hist['mAP50-95'].max():.4f} at epoch {int(hist.loc[pk,'epoch'])}")
print(f"Best mAP50    : {hist['mAP50'].max():.4f}")

In [ ]:
gallery = all_preds[:8]
fig, axes = plt.subplots(len(gallery), 2, figsize=(13, 4.6*len(gallery)))
for row, (img_t, tgt, b, s, l) in zip(axes, gallery):
    row[0].imshow((img_t.permute(1,2,0).numpy()*255).astype(np.uint8))
    row[0].set_title(f"Original — {Path(tgt['path']).name}", fontsize=10); row[0].axis("off")
    row[1].imshow(draw(img_t, b, l, s))
    if len(b):
        f_ = Counter(IDX_TO_NAME[int(x)] for x in l)
        t = ", ".join(f"{k} x{v}" for k,v in f_.items()) + chr(10) + f"conf {s.min():.2f}-{s.max():.2f}"
    else:
        t = "nothing detected"
    row[1].set_title(t, fontsize=10); row[1].axis("off")
fig.suptitle("SSDLite — predictions", fontsize=14, y=0.999)
plt.tight_layout(); plt.show()

In [ ]:
report = {
    "run_name": RUN_NAME, "architecture": "SSDLite320-MobileNetV3-Large (torchvision)",
    "base_weights": str(OG_WEIGHTS), "parameters_M": round(n_params/1e6, 2),
    "gpu_available": GPU_AVAILABLE, "gpu_type": GPU_TYPE, "device": str(DEVICE),
    "host": HOST, "epochs_configured": EPOCHS, "epochs_ran": len(history),
    "batch": BATCH, "imgsz": IMGSZ, "lr": LR,
    "train_minutes": round(elapsed/60, 1),
    "metrics": summary, "best_epoch": best["epoch"],
    "per_class": per_class.to_dict(orient="records"),
    "latency": {str(k): v for k, v in bench.items()},
    "saved_weights": saved, "artifacts": exports,
}
(RUNS_DIR / f"{RUN_NAME}_report.json").write_text(json.dumps(report, indent=2))

print("=" * 64)
print(f"  SSDLITE320-MOBILENETV3  —  {'GPU (' + str(GPU_TYPE) + ')' if GPU_AVAILABLE else 'CPU'}")
print("=" * 64)
print(f"  Trained      {len(history)} epochs in {elapsed/60:.1f} min")
print(f"  Model size   {BEST.stat().st_size/1024**2:.2f} MB ({n_params/1e6:.2f} M params)")
print()
print(f"  mAP50        {summary['mAP50']:.4f}")
print(f"  mAP50-95     {summary['mAP50-95']:.4f}   (best epoch {best['epoch']})")
print()
print(f"  Speed        {bench[IMGSZ]['fps']:.1f} FPS @ {IMGSZ}px on {DEVICE}")
print()
print(f"  Strongest    {', '.join(per_class.head(3)['class'])}")
print(f"  Weakest      {', '.join(per_class.tail(3)['class'])}")
print()
print(f"  og_model/       {MODEL_NAME}")
print(f"  trained_model/  {', '.join(Path(v).name for v in saved.values())}")
print(f"  predictions/    {PRED_DIR.name}/  ({len(all_preds)} images)")
print("=" * 64)

## 17. Try your own image

In [ ]:
MY_IMAGE = None   # e.g. r"C:\Users\you\Desktop\site.jpg"

if MY_IMAGE:
    raw = cv2.cvtColor(cv2.imread(MY_IMAGE), cv2.COLOR_BGR2RGB)
    img_t = torch.from_numpy(cv2.resize(raw, (IMGSZ, IMGSZ))).permute(2,0,1).float()/255.
    with torch.no_grad():
        o = model([img_t.to(DEVICE)])[0]
    k = o["scores"] >= CONF
    b, s, l = o["boxes"][k].cpu().numpy(), o["scores"][k].cpu().numpy(), o["labels"][k].cpu().numpy()
else:
    img_t, tgt, b, s, l = predict_one(min(20, len(test_ds)-1))

fig, axes = plt.subplots(1, 2, figsize=(13, 5.5))
axes[0].imshow((img_t.permute(1,2,0).numpy()*255).astype(np.uint8))
axes[0].set_title("Input"); axes[0].axis("off")
axes[1].imshow(draw(img_t, b, l, s))
axes[1].set_title(f"Detected {len(b)} objects"); axes[1].axis("off")
plt.tight_layout(); plt.show()

if len(b):
    display(pd.DataFrame([{"class": IDX_TO_NAME[int(x)], "confidence": round(float(c),3)}
                          for x, c in zip(l, s)])
            .sort_values("confidence", ascending=False).reset_index(drop=True))
else:
    print(f"Nothing above conf {CONF}.")

---

# 18. Live camera

Live webcam detection in a window. Missing-PPE boxes are **red** and counted top-right.

Capture runs in a background thread. That matters more than model speed here: a plain
`read -> infer -> draw` loop blocks ~35 ms on `cap.read()` and the driver queues frames
while inference runs, so `read()` keeps handing back *older* frames and the lag grows the
longer you watch. Reading continuously in a thread drains that queue and the main loop
always takes the newest frame.

Measured on this machine: **22 FPS serial vs 118 FPS threaded**, and the display now
tracks the camera exactly rather than falling behind.

### Controls

| Key | Action | Key | Action |
|---|---|---|---|
| `Q` / `ESC` | quit | `SPACE` | pause |
| `+` / `-` | confidence | `[` / `]` | input size |
| `M` | next model | `B` | boxes on/off |
| `S` | save snapshot | `H` | toggle help |

In [ ]:
import ppe_lib as ppe

cams = ppe.list_cameras()
print("Available camera indices:", cams if cams else "none detected")

# Load all three so M cycles between them live.
live_models = {n: ppe.Detector(n, conf=0.35, verbose=False) for n in ppe.MODELS}
live = live_models["ssdlite"]
print(f"loaded {len(live_models)} models — press M in the window to cycle")

In [ ]:
CAMERA    = 0        # pick from the indices printed above
CONF_LIVE = 0.35     # adjustable live with + / -

# ---- UNCOMMENT THE NEXT LINE TO OPEN THE CAMERA WINDOW ----
# ppe.live_camera(live, camera=CAMERA, conf=CONF_LIVE, models=live_models)

print("Ready. Uncomment the ppe.live_camera(...) line above, then re-run this cell.")
print(f"  model  : {live.spec['label']}   (M cycles through all three)")
print(f"  camera : {CAMERA}")
print(f"  device : {live.backend.upper()}")
print()
print("  Q quit | SPACE pause | +/- conf | [ ] size | M model | B boxes | S save | H help")